# 05_value_guided_planning.ipynb – Value-Guided Planning with JEPA-style Latents

**Goal**  
Bridge representation learning to control: learn a compact latent dynamics/value model and use value-guided random shooting for planning in CartPole.

**What you will build**
1. Collect transitions from random rollouts
2. Train a lightweight latent model (encoder + dynamics + value head)
3. Plan actions by optimizing predicted latent value over action sequences
4. Compare planned policy vs random baseline

**Reference**
- Value-guided JEPA planning: https://arxiv.org/abs/2601.00844

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import gymnasium as gym
from tqdm.auto import tqdm

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent

src_dir = repo_root / "src"
if str(src_dir) not in sys.path:
    sys.path.insert(0, str(src_dir))

from jepa_tutorial import ValueHead
from jepa_tutorial.storage import CHECKPOINTS_DIR
from viz.planning_rollout_viz import plot_episode_rewards

torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Collect an offline dataset from seeded random interactions
env = gym.make("CartPole-v1")

def collect_dataset(env, episodes=80, gamma=0.99, seed=42):
    data = []
    returns = []

    for episode in range(episodes):
        episode_seed = seed + episode
        env.action_space.seed(episode_seed)
        obs, _ = env.reset(seed=episode_seed)
        traj = []
        done = False
        while not done:
            action = env.action_space.sample()
            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            traj.append((obs, action, reward, next_obs, done))
            obs = next_obs

        g = 0.0
        episode_returns = []
        for (_, _, r, _, _) in reversed(traj):
            g = r + gamma * g
            episode_returns.append(g)
        episode_returns.reverse()

        for transition, ret in zip(traj, episode_returns):
            data.append(transition)
            returns.append(ret)

    return data, np.array(returns, dtype=np.float32)

dataset, mc_returns = collect_dataset(env, episodes=80, seed=42)
print(f"Transitions: {len(dataset)} | training reset seeds: 42..121 | action seeds: 42..121")

In [ ]:
# Prepare tensors
obs = torch.tensor(np.stack([d[0] for d in dataset]), dtype=torch.float32, device=device)
act = torch.tensor(np.array([d[1] for d in dataset]), dtype=torch.long, device=device)
next_obs = torch.tensor(np.stack([d[3] for d in dataset]), dtype=torch.float32, device=device)
ret = torch.tensor(mc_returns, dtype=torch.float32, device=device)

class StateEncoder(nn.Module):
    def __init__(self, in_dim=4, latent_dim=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )

    def forward(self, x):
        return self.net(x)

class LatentDynamics(nn.Module):
    def __init__(self, latent_dim=64, n_actions=2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim + n_actions, 128), nn.ReLU(),
            nn.Linear(128, latent_dim),
        )
        self.n_actions = n_actions

    def forward(self, z, a):
        a_one_hot = F.one_hot(a.long(), num_classes=self.n_actions).float()
        return self.net(torch.cat([z, a_one_hot], dim=-1))

latent_dim = 64
encoder = StateEncoder(latent_dim=latent_dim).to(device)
dynamics = LatentDynamics(latent_dim=latent_dim, n_actions=env.action_space.n).to(device)
value_head = ValueHead(latent_dim=latent_dim, hidden_dim=128).to(device)

params = list(encoder.parameters()) + list(dynamics.parameters()) + list(value_head.parameters())
opt = optim.Adam(params, lr=1e-3)

In [ ]:
# Train latent dynamics + value shaping
epochs = 80
batch_size = 512
n = obs.shape[0]

train_losses = []
for epoch in range(epochs):
    idx = torch.randperm(n, device=device)
    epoch_loss = 0.0

    for i in range(0, n, batch_size):
        b = idx[i:i+batch_size]
        ob = obs[b]
        ac = act[b]
        nob = next_obs[b]
        target_ret = ret[b]

        z = encoder(ob)
        z_next_target = encoder(nob).detach()
        z_next_pred = dynamics(z, ac)

        dynamics_loss = F.mse_loss(z_next_pred, z_next_target)
        value_pred = value_head(z)
        value_loss = F.mse_loss(value_pred, target_ret)

        loss = dynamics_loss + 0.1 * value_loss

        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()

        epoch_loss += loss.item()

    avg = epoch_loss / max(1, (n // batch_size))
    train_losses.append(avg)
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{epochs} | loss={avg:.4f}")

plt.figure(figsize=(6, 4))
plt.plot(train_losses)
plt.title("Latent model training loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.tight_layout()
plt.show()

In [ ]:
# Save components for notebook 06 under the ignored notebook checkpoint directory.
checkpoint_dir = CHECKPOINTS_DIR
checkpoint_dir.mkdir(parents=True, exist_ok=True)
checkpoint_path = checkpoint_dir / "value_guided_cartpole.pt"
torch.save({
    "encoder": encoder.state_dict(),
    "dynamics": dynamics.state_dict(),
    "value_head": value_head.state_dict(),
    "latent_dim": latent_dim,
}, checkpoint_path)
print("Saved checkpoint:", checkpoint_path)

In [ ]:
# Compare both policies on the same reset seeds.
@torch.no_grad()
def score_action_sequences(z0, action_sequences):
    actions = torch.as_tensor(action_sequences, dtype=torch.long, device=device)
    z = z0.expand(actions.shape[0], -1)
    scores = torch.zeros(actions.shape[0], dtype=z.dtype, device=device)
    discount = 1.0
    for step in range(actions.shape[1]):
        z = dynamics(z, actions[:, step])
        scores += discount * value_head(z).squeeze(-1)
        discount *= 0.98
    return scores

@torch.no_grad()
def plan_action(obs_np, horizon=12, n_candidates=256):
    z0 = encoder(torch.tensor(obs_np, dtype=torch.float32, device=device).unsqueeze(0))
    candidates = np.random.randint(0, env.action_space.n, size=(n_candidates, horizon))
    # Keep the seeded candidate sequences; score them together to reduce Python overhead.
    scores = score_action_sequences(z0, candidates)
    best = int(scores.argmax().item())
    return int(candidates[best, 0])

def run_policy(env, seeds, planner=None):
    rewards = []
    for episode_seed in seeds:
        episode_seed = int(episode_seed)
        env.action_space.seed(episode_seed)
        if planner is not None:
            np.random.seed(episode_seed)
        obs, _ = env.reset(seed=episode_seed)
        done = False
        ep_reward = 0.0
        while not done:
            action = env.action_space.sample() if planner is None else planner(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            ep_reward += reward
        rewards.append(ep_reward)
    return np.array(rewards, dtype=np.float32)

eval_seeds = list(range(100, 125))
eval_env = gym.make("CartPole-v1")
random_rewards = run_policy(eval_env, seeds=eval_seeds, planner=None)
planned_rewards = run_policy(eval_env, seeds=eval_seeds, planner=lambda obs: plan_action(obs))

eval_env.close()
print("evaluation reset/action seeds:", eval_seeds)
print("Random policy returns:", random_rewards.astype(int).tolist())
print("Planned policy returns:", planned_rewards.astype(int).tolist())
print(f"Random policy mean reward: {random_rewards.mean():.2f}")
print(f"Planned policy mean reward: {planned_rewards.mean():.2f}")

In [ ]:
# Visualize return comparison
plot_episode_rewards(random_rewards, title="Random policy episode rewards")
plt.show()
plot_episode_rewards(planned_rewards, title="Value-guided planned policy rewards")
plt.show()

print("Reward improvement:", round(float(planned_rewards.mean() - random_rewards.mean()), 2))

## Suggested exercises

1. Increase planning horizon from 12 to 20 and observe compute/performance trade-offs.
2. Replace random shooting with CEM-style candidate refinement.
3. Compare latent planning with direct state-space value planning.